# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJztfQtb20iW6F+pIbcbu9sYQ5KebmaZvQTcCdMEspikd7/A1SfbMqhjSx49oBmW/37Po0qql2RDZnd291vNNx0s1fPUqfOuUw8bkzSZxdf95f3GntiIF8s0K0SaXybyzyyq/izLeFr9KOJF/SW/KYt4Xn+7yaJwGifXl8ksSxeiuF/CDyG/ni2LOE3CeU+cxHnRE0fxBP57kNzL0suwuJnHY1X8A/y8TC6TF+IoKqJJIYbJbZylySJKij3xNk2v55E4TOfhWNzm4iSdhDCQ41FweHZy8Ebsw1z62GI/+h16yzuXG9sw4wIqX250RZhMRZIWbqHDvUt8NrrY9TSaiWByE02+BHEeTNO7ZJ6G0ygLpnHWmUAb8TQsIvy1J/Ii64qtP4txms73LhMBz+UG/u9TlMWzOMrFXVzciKMsvo3Ezx9HQzEJoWkRJ7fhHNsB2Ii7m6i4iTIRCmgT5pxm9yLO4SeVEfUIxF2afcmX4STqcy/cY5Hdy77xiWc0R2OgIs2Mice5M5eu1gQ+WVSUWSJ+Dud5VH95Id6UecEzqQc7BzDS+qti5ojwgRXIEBz74nPUn6d3UdbpihkMC2GB48I23EFd1a1Ev0+iJeAD/QNgW2u8+sh3+uIQV5W6hR50cC/C7EuU5XVx/B4sNYT6LY0Tc3A9WGosRmijQV+9xYmpaSPmmcDnDmyou4CrBrM2BGXLV9YCtIPP143VAk3tr2VURv3pmKdn1IChXG4soiLEt/3f8jRxC3l6lQt3kZWR+XEyB1SDbeesAk8QwS+LGCtQDbX66IzUXY2qL3tBmhdFH+L62F31c+W2uMYSNXTqaYxAsAixpLYWdl1aM6QqiPy+Eg1jMBZN32S7+ibLi/Aa6ML25KZMvuTuJqMxykKr9kv7Pqwa6TbQMXekL/WRIs8CViIW0TQORV6OZ+kcCG4uOll414NGZnESTc19Hib3nS/6oLEhevH5cgOqXW7gyGTVy42rtUf2yjeyZZb+hvxQAlF0gJ5HwHa+5O6oon6UTHPkPMDb+vHyPhkj+6sQUg54zQG5lLUFUc3CzEyzKE/nt5HNSolxAgM1+Cb/fXSfhAuc9hyYC/D4AqAreT+zUo0nVkS8B+u2RCECpY/8Bv6YlEXe4xb76sVWEWbXUZFvje+3gLnS2EqcRI+QLr8Js2gq1PLf3cTQ53UZZiFIERG2PIM3EoehqDgdfhqei7ssLkDKkIOBURepOeAsTYu+O1HJrJUAowFyHObMumviF45z/LfaCvAVwBR1ggDHFATdrrMOxq5RTRLjqtei3jtq1YlXfshiwLV7QZLQnjgsswzwhkQQBEPNOzswiUVaAnymokxwSfSZyzFN7qb+mcBawLeOGju0BT/7MKasUBisRLjtKTa4rSQ5bBKBhxvObtiuBHV6Tu9Oy+/vj9YvPCJMob+BdHavTBGsWYC8mzbsO/hSL0BFR5sQt2OglyokZCEBYiVJyRKs8nWAaAjr4EymoRcNTS2ZWW9Qn4/LK4mIwYBNRthUv2onwN5szq/X6mGrXbem3FIm9+DmfB0RcgIs4sSWP1atZHubclG50NpS7DLM8xoJgEWNojCb3Ag/orJyMY2ipa5bjEtNGl/cUw3fmlfY3rTGsm778tZSiLa6qibM4VhpOpGtNdBozcZgymFf/BxnoGMgKPaA+qfMB2lMSHgqHUjOHgV5R7LAh2rB6O41ludZrFk5n7syppxBj1vwo5mJYtwOk6ZGnJGF2nGGC+ksWMJmjPgAizh1gRNOSOVLExIWmH0J5A4ChKU82oqTPEryuKgp8hPBlKTZIqAG92GcfRCROtnm58vL/PJyK7j6frMnNsWmBBeQ7yxeAk1XsnDDLlUtsshUg4olpwlp+VOD34MY1QC7Zy5j21I29ORZKfX6Sbv7VbW7mZUwKZdInGvcxbd5Te7TtIP1Vp6xjY3qf8fNpbf7D91h9uf/6kj+D8LUw4qwgDgflvOCrHY4I10IcZCQh9jEd7Z1UHRdFWKtelK4VOOTAnNjVT+0uRElheltIQiOzn49PTk7OBqeB0fH54QYfnXmMoG5W5L8ydnhwUnw69n5L6MPB4dDc2xFtFhi7xHoSrL8m4PRUHbzVKnf7crYc6plRjppEsUfPIouG11HRVZOAA5AjQz58khy7Rjp0/nBr9YYqQcTUKT7ohIMLZ8Pfz4+HR6tWUkpzFrF92efjocjX32taay8SG9jEsfruhef1qhX3AZ5lNlVzy7eVaveVjtF8y3VPDq4OFhvmpXl8P3w4oCq/WV0durUk+31XNOaXLHh8iZaRBnsTTKGi0/vzcVitGhYMgtn5JoFMTcvq9Jc162Ni6eUyYvzg9G7YHRx8Pb49O2aDRRZmN8YKikXagKsp4kKsvytFb5m434oXwzff5BdW73Bep8ff1KwMddYOjFi5BLjEnXjtx8+ikO2rolvQbsjm9iIZ2quWQvIXETSYXX47uPpL6M2kGufiBLQKDRoyRbWg7RW+/Dk4+jCv1l06OoGW1UFmxyee/vUWu3phkqt9vsD+ufn4xOH5Fm1DWNoPeHmgbvIoQ/frN4yCacfeypmgZYJeVuyp2WjnUK5Q/L8lRn5nCSqBEcfzw8ujs9Og9HwEHrbfTXoDyohAERUsQC9GK1vS2jhS3Q/y1CUyaNr9MeJTgqywwKIzjichwloZOlMsHxQpHOgSfjqW9BS7nQH2PH2mcLU4GSIXOni+P3w7OOFHMPLgRzDC/G66n4egRIjpnL0orPMolsYQQ7dAfMU4RhkQnYS5lgKd9SrrQWasxFFQd4Nk3ySTkmM5aUK3g0Pzi/eDA8urP53XkP3L8R7UCkFrxJMazYHnkSC5zSaR6DLkmgnbkB+L8ZRWJA8vPMa9u77g38NDg4vkCyohfrL2ZsRNPyqhqu0byrDQ1oCgONlhJ3sCZjb1riczdD+WC7RkvhK/JaO0RgZVfZ5kc9TnGu0wEHhOIFsfBgefjw54L4JvrDbT07kvF7xvFiBzHGpBt+gVfka5JkcTfVzWMcI5gHzlGCLcwFjuo7HIAeQPX8ZTcp5iEokSIDAc+bRHCRJeMk4haj3KcrGKSzJSXpNhO1bcRSNy2txmhbxDOQqLJmLC/g6B4nxaPjm49vg5OztaI98qDBOFHQidvqicRDkx+HpwZsToLJVWRZcdtAQ+AeUqAaE+KN/A2i/xwK0ecg520aL8nsA5AKk9+t63wDhRyspuaHzPVMGUv7mygQ6SZf3igX8y8fhRxjim7YOa99Ztembqrmkx6iMYz2WFr6eCOd34T3gCox7TgMcjY7E6F9OYsBTlBERhWAdYTugFcaYE+1ILLMFNWlSszCeg/yX/0mQVHMXQ6vYsjlNn6yrzcWcnCHhaqXsJtlbUC7ZsWNK10l0Z3rcT9MkMjwHus9gCTCE3YEoLe2jwFrD64j0lhzhEQpoEUpOY6ALtxEKvD6fguFnv56nQOiEzYKlbAV/6FKhK7rW71gkrX9XcqbZT7X6hgjTE5XQ0RMGO3e5P7yrmYXRuIdv1YU1FgRdmNuqZy1bT+gLWGm9hZAr5g89kB9dtU814mhdtjqkWuDiT1ZJqNJz1RK98nNUE73+M9QTvfrTVRSC7tPVFKz2FaoK9doiLOP3Z4u9WPlrRF+q/3QBVq/2PCFWb+H5gizBztylT+B79co8mYdpu93jwWyl9FiA4nDYLlZVejZxrZtYSWV7Gml9CgHt+Vab+71qNa4CNBfhlwjIVd6Z9tiVEaRf9tHTrpkN17TJ5VFhcUeAsJdrSgtBkgNDB3o8AUEPPbrTWtfksVwm9lJ85TL85y5Bz2sA6wnD9FH9rMwZPeEYKFShenC2Vly90cfUqApeJgoxTKRYAyFakEEhwgvU6Sbkl5/cg2C/CJc3IL7nIHP/yorLCQh9hVRK1JKq8aGqxV9Ozn4NQHfAVXXLDFBfGQBqTGOOw/h4enL8/vhieITay0RGBYCOtwXagSZFyV4luOv2GvQjkv5kVeB1cipQvoou7VcT7PimA2CDQQQ0iMDbEkqMOBmSHEWZoESMCvJEAyJpOTgZ0GdBbs7iyZd5pM8JCoO0mMOCrDVKc/pScr+IF1FaFqROjghov4bF5GaaXovzkoJLqskBbp7bCupPoB9rRerVu3h3Phy9Ozs5Ct58QFDvDAbiO/jv7qs+rSL+/uXNdu6vbCrBu9SJAWgey/nw4hw2OxoLBLl34yIGefJ7+AkSnPQhvQQRGxRKERZoXS5ypbFsLaIFun4vwvyLuMhCqXIA0AF88y2MMhZzlNcXKbSbZuTGLqCwinKjvwPUVQygg172BQ3w+DmYUCwKEsUB9/uWRV6pzKPOnaXzXMM3/hKEk4Jd5RwBxRLfWh8bB8QqTR4ilP4WkcWejPf4n1qXsYOgRrJ8LlQFgtEkS/N8awkKOPxawEQWwKPjcTyPi3tCJjMkJJxFhanAKMkcO/f4W27jaZSqsh4v2OXl9nf/vHe58U9//verzR6NtUfluv4ql5f591ROVAWV08xwueCXz3uAclc4Qpgwgacr/oxoKFB3pCIKmuN7JIABW4rZHNMhVwhKLAFsTPLvtMRmS1cLNWMHll1gq+JNrOjt+xQdsCA8syZNIetGBVA0U/o+V+b3qLLI16EKMwxpkHyqQDPOMsoWQGISjBXLYFNgL3HBjczv++LiJs5FZeXSx/bXErYWYnFeLogzNIeVuVCxtTHpJ3YLehxyWgBgMzfzsNQmcadMYpAmlbt1pmz/D3jwoI//edXp9m+i3z/v/Xj1GDyoIaN9hvaQZ9CPCgD4ECwBQfLCFmg9g9QGow3xBa0/2v/MRU9x0XnBKRyQJOq6Gh+Q6OOqekbZ00Zm9PURWFLyhXGMG5/G+RcjrMxaN60hV+RktOp4O3OCrm1xQ4QgG+oyB9DiogOL9JmwkDfPFTCuDN09eyCJlHMOx+PtKWgxiWw9uCB43BMP0aMRtu1Kze5029HUnbqnvFvcG3++lihutWDEvE7mUZiUS+6V4tUCoEfhPbEAEPqq13vVCZnPUOBKyildD8EaTcIkd6NKRcfv7O7KEGzoVIDChjufVEYjgHUyAZqfFGQ0UzGsaCmIYBnroMI6SFbEeV5y3DiZDq9vUthek3QJfF+SN4qiBd6jETZocSHKHHm9h3j76VcNITRDk+ymju94dE1lPMK/65A3rQ0KdF8ZAOchjZ4gONNSZSJuazjcSsnebPkFmqFBkklybSmnykSN0FB2Fn1NGc2Wqua+rl0Da0apAQp3ssuN/8fK/HeXl4Af/e/+D3JpKHH89vTsfHh4MBp2e41VKwsP1ka0aqt/Vc8ILezI1/Iou2VfjvQ5yA2zpwoez2or1TQFiOLa3ISwrsoEgWtqGJqw93xSRsSOx0j3a56NSOBbL6SLAbT1FJNHM7LI1pwAX9z+bidVcFpDH7Ifqw/ZWNc4I4FRsSAw1h//LAZ+0lhx6nqKTQxaPZKdobtjRuvPnfQU8DxVanZxyOt6Jc5pbabyUFu1iLQ6GjUDxvog233UQtKrtcIVf7rpccWSYSXvovl605atubOWtaPOmlePP/+Hrx9206tB+pw1NGbfvJBYrOb16wsaVa9SyOCTCLXCSFudCF8tTch4MKwZTVkDxBcrohU5OM4Tb94YPuoEmRNgq3ZsXHIDzHFEUBLHYxJsVwiCL7DqxeSmw6Pw4IV3qdDhgCdhiG3QkoRZEc9AkUV4YVOPDsri49Wvqpm55RW0v98XO57WQCX+UjFYVdZEb8/wD2XBcikeZKVHxQHH27Uc49HI+KiNcpOelwlPXjIYPBmxSKclFJdHjmFYWZkQhsXSp+pzb66Q6KA7YH8b8/T6Osoaz1eDpuA/U43WW+NF+6lq6yx1rxIipYWJKm+hGAaaxHXl/vxWWT8ukwDeu3aLc2W4gIUfl9cBaFrjeTStnPNIYZA7D/k9SokqJJT2WJjjC6iJvpkeBlMCIwZSgN7+KepR0HIWTQD7sH/YuDiRzzgPlH5pLldX0tYTsG3ybfDueHRxdv5vHBsiR16J9n75ubK8oK0cx9MxZmKK1zwZFCdA36I/b2UQA1VFCFIoQ0gGGPIqMwblPvewCTn+RCcVKohrmG8WhsHj6NRYtb1Wx0R89sZDXFFk6Q4eopzZjZIBhYIkWrUtRizOD+BQL8wZUPcHnflmic+aPgxcG4zdpqXxW2nOSRbOGeVgVv41wRAVHoOxFlIls0dZI4XCoY4Hh0zsUJ9An8nxZFWItpsc0Bbt3iRgEudA9EeUFp2of91HKmOcgvNiioHIqzDFKAwrsJSZEihdARoEwyJACpIX4WJpH66U8FBUpl/9kaR3HbLKzfBn53Ljm3d737zf+2ZkJEMA0pcFaHJKoX+MUuqYMPo1XtKh3XuATYKERFxe4qmvCRpcmXyBSA78DWNQ5oKCqYieAlDlrjobCWXZzFuyGwRI0wJUVXGw0Aug/PdsYhTfiZ9e0w96vb4fheaotVtEvxdsPrSssqTUUhmKIQHlLi2BAIdlkQL0ZexJFsE8JgiDMomSSTolivIxiTH+DGSYMKPTefPoOpzcCwnUtikDz+hzV/16fF3v99m8zG865sw7suthQv/NshTo7NmI/lh1JmQCeAbCKYgmWafuBiVfmhn533tssGANuyxmWz/qhIYGiKDFQSOrgX/6VDkCMjcByQrHke9jgAXCDQ9f9qeR+m6JGw4oqqabCxoweSKJmke30TzI7xdjoMr0w2e0n98ia6TP/XK5rM+CoDUbvyFtPj79+exyw2d2vyx3d17+dFnOosFMgS6a63WJ8jZVnv74cgr/nQ6mTmUQZ2CT/HpwfsoRc/jX8enbhuMWMI4fwkHbOIbn52fnjZP446uJv9ro4+HhcDRqrjh4bdFtej3Y3cXXcn9Gi5iotrYMPVGE1/KvBfq2ryP5C6jIBAglipWV6ELmOZ9d7RDND4qfYDcYXMGfDpZIz3KpO8CWjysXFkdo9jRygIIW8YOYWAyfJ6cx0Ge50z3Wrkaar8z49WyoTYKAgRmVacwS2yyN1rAu4VOgfcjDPEx71j0UejCbutyoSsOyiuqYff2dBonf6A/nM6wcVQyvnU9yKfGz/NOtjZIyVkcWRoxLG/Kjxi00WbMf0mJ2PKocupz0oqh+29KnBUuj5WW67AwMUwXMK+DwU9RqHuDn4xULZ/Cnksh0OslrUJAMyFXyxyvx4BKg7qN4kK3DXxJAj7xRVGMvSH3KgM1cp4oZE8+VtjZCJfyQRHeEtuRm0sDmY/h+HlyPmxlvYtlsUBw15XUTjv7cKrqdwzn3ZDTXXW3+oO2VLnGNjapIEUMkjIqX7Vf8q4tWiVnTSdN+29T1kmskdNG5TYIh0vcdP1EzKde7+PpmC5YxzdD3m2ix1WKB6miB9vKODAym5Y4w18V1mE3nGO6dzqQczaqZKZ7WtFaxLaK01XgM+qqgLkXrcoJRRmvNYcRl1xx98xAr9rL2KFntWGeMRyD8xKhETePwOklzEPJow3QUTFMUYWMFzVobaR6uJNltg2VWpUZ7F2bJWoNVdjLazSZY1wamEhbWhCRJb2sNjuTNrxqalD7WHhsIrjqV7lC8zx467Dk2hqQ4n/3Bq4ZyS0wtVZIiPDZRnZecQiOmGN/I16WUg+4hg+983uIxXjXNgURd/0TsGBI17ppKLcp5ERMbwCAMwJN05pnVejOo3Vze8TsijP+EeCXtnbL0lBhnRKDhNJvCwO8jtOfVVXEOdtorO7eRzWZMfhx93oQ3m1eKKavfPs5c9agEiKoFJQFhOxarhs/0e/PK5NfwXiIuVDL4RS35IiNh8zN129Vdyhmjsrm1iNnnmmyKsLyJc+cAQ7NxQZdmqB9lz7yJ5qDQyDinp5s1yS/izShpJZEss/k8Hvez6K9lhAGp5msiM/bLJUxaS0pZjpdZigxlTROqx3jKOSov6xEra5j8ycZdfW9eL8sgTmYp21tMOmKs0XGCh6dA3uZYEXEDnPgO80ch4p5+Oj46PqADoljK3IPYvCmAX25gtxQSBcsB6w/1OOATqKNe7DYLF8FijKUGxgeyYWdcW/Xz6LM9YJjivgbbPqddgNfLsuiYm+Tz5UZyGwOj3MoXMSucW1uwnNn9Fox3H8fbYwTtU3Rgj0cRyHhKVYMJ1v4kv+0l6U2EsczwR5lgIOvGlaUKID3cN0M3rBgzfJaYTAoJxlJ9Y/8L5SQqi36+nMd40KyHeZwczYBqg0qwL17aNlNYms/6alyRXQ6Kfx7YyRO5aLUiWBKdHlx658p2HXJxtU51u7tXa5q1NJqCjWlG1wR48CQIb0GwQVGlwfxK2adyhMHpp+HpobiJbifbN7s/vGJxGQ8nIvrKlGskCcGGAwFIkopme1Y7TsGsZ7PFMrqW+KB6w7Wvl7uHxjegCvtaM0fDT6cfT058NBUHz/PmNIM4BLJX4YSsDyvh6wv3wZxLwazMWbPoaN6JxvNqF1mImK9cWJrnFONS0WJQW3vRskgcu4opwe+gduJpTH9ipZYFqLaDhqOTUj8rqXStWs2qS3JKPCxPSdjg3z+4FeG17aekXhUPxe+Wk8E7ALNctWkxLhJAN5V7szlwjE+bPTGrpxa5s/Tok2um6ay34Rom6MkcY+beEbNlUu4xVB2UoBilCzwtjKzinEKQMxXrn6B5uXZk5vEUD/JyqmTm4kKljJQ2qCN59jEX70MA6i1mvtQyEfVA4gjjhTqEvKvH+1OoMJ8SVqYf3qg5NVNxN6IdbAnLpECKUUToU56qY+S5kTVRH4HHVoZWmcOz08OP5+fD0wt5MAOD0ZXRgYyFwJNBbQg6eTSf9SRFCuKp31cI80xB0MTTsWkWgHDRVIw9v5xbK+AI7rbYPXxwAH0lJLAxXZMZtHIYaodfkDLqdUz20pdGcuBVLDMwkUSbef3lIsrnYf1BcTyid6ewRLpwS31V8IHO67+h/EydVgseqgGqbEePDYG6TuuIl1qQL+O46DyY09xUswS5uOu0wSfe/GtjLZ6abz/jPy43KGEkxh6YBZl8+SuzHsCOZGMgcn8Z/nCrCB8UCOh8/h6LhMDoayHT1jZxDg+DPYljO/zHY0ujzacNjCozMlwEvEcDmOgc+YocEYm9s3kaFtT9o2k9POcMR0C4QDCaxIXMmCSzymFISpYiBdD8vlXEprY9OF+C/VbB2E4TqXCksYLFL1QKYCvE0Wyv534wAnyaY0J7soXuk5swU7NV7ZjNWOIhHe+g9MvVtDxsxWFsk6bMX83MjWDHDn3/SXvb56Y/WjCP/azJFfExo6dfCDJmAjan5fUNiIDi/OD9Fka+q/zRe5SAYHsa3W7nNwtKcqyEV0KxmzBH6fz1YCDevwFpKgKxFE9d6OR1odzmsFyyIYfKUIRygBI69b5vJ5P35dfjdruehNXVt7Us3jBVkoo5VA/7D0o0E9TNeNEBq/VpxhIAfN5L/tOwBjRZPhaAx7OsLMMyQ9+ispFrJgBvvFgz/Nyc7vhoySCeMTLljmYcxvHZSZmI81lJ1rqrJrIG+v5jBueB1j9mILpPxjMC1w9Ty2MqyUZQ83FqghQjg59rM9WlPnyGv9+EmNn2NqJIHEycKXOuk/SqcfJNTDVfHZEX4yicYJwpxZloHez0FSH0HcFn6YA+N+ZnqtvCdM3NCbL72xgmuy1zB2xrg2PCAVoMpVPqtxSs+8LE9STVm8f9tJz1DdkPv1undcxKqnKnOy01pIler+XXfZXoVKY1tVvXGed6Tf6Mtv9cz2XjXEpiXrWCWjMfqmMtebq9iPMcE75TMKnKmt1vRsP66oEl5wgiD4KURT9fmYIUXjlCOCQTcHP+GVOhxX0qk4P6UGyj3rzez3aq0KUWsqN2P/6QYTqOprz0UDprjkppX3bN2QHSc0KkLKqSQhnTa2inKbOlFKc45YiRbMPNeqHnun9eN/0+t7t2d/rUYQ/+kqCZxkRhvqyAvBd1clQZUazB3KXOKtYY+zYCVdRTRx03yLstkuwahS2ZdY0aKzLGq8cj5075UFeiz8kv7PqOazQljCdEaMzj3iwPew9scS/ytqNcS67ub2PNZMjqAVLtudSGc9q35amWULHETGysLQEwAaZ9h7DIhw2tvyG+0i6mb6ajaGtaoqaJtkx5NoysnRnm+NVk9Cgh80ih64Ly4K6pDlqOwcnSUK0k5TaHh4maPWmmJrYFUOZ0VjdP4KC8smKU9MPptINlPbByRq2WRJY3xy+ZB3Tn1LP6Rt19UmY5S0ljz11dfEdAdSauMjbry+vabYV7yZDfYMtDtWb8XJNx1X2z2bh5JG7ZNc3HcpXXMCHj065przAl4/NkxbnR2G1czfE0k3cdI8W1KePZM2Kj8KEDqfvk+u2jXaEza5g5GuH2qbhM9FirB42KJkxX2e5A5urA313zspiboliaAa72Q4HnAXdetWFYC5vrvhBnVdrF+DrBYLqd3T/2B/C/HbHNp/bpSHSY0+lnIGgy3xeSH6QZGWgjN7q2onI+tNL8dSQGAgrMoBqQutBLTZcswNUQra8rWUgjb61hIyO68LCYkVmR11UqYn8Qo/tkcpOlSfy3qDpb4kkH53v46Gns2++rfstt4eNcGqCrDnSDjnHy1r0q0X7aSYIG0GYjXNVVeztPoB3qMY1v9qOS4Cus+Dt4v7Rm1cksfEV+GsaIOi1OZRSoo5m06rPLjQfTYfFYXW03s1wZJlGQr5F4kWVfd2x4G9RHifu2TCZRUKR8+E+6ldD2VuYqzezlRjydRyblkWF5HCE+SefACgBiW2Svk9mIpHGC4sY9bjBdxe/bqqgktE5odjUbDPAw5+dEUlfemaps9cYpq0edtDipnHpaGIpdrY6HcGoxeKkS/eVGmANVCaoMzFY4uFM6i67xzBYANXAK12U1v8s4mEmPBA3awVU9d4m8dMtRzFyy7UsSt9EzVQFFE31pJ80MkvY0m8x+T+i1Jem51XfdtSVuU3wC8BYJlidHfBdec6Jd446WwlajuK5cu9ZgcK5Pc3qGoEPSzbRcLOk20p4AehAnmEplf7eBZs/ck0irwaKBZ5YD2+xAK4iAaaeJjz3bD7NmZV0GpahECt4BbhlkGLidRYGil1k7nbSijvCxCVx1iXJxExZ6dNFmXlFlk4Qyp0Y7npeAVi0fzzDb1l2Yy3zuU3lsc3xviGbEB9mIKPO/JZRE3siLsjUBma7g/F3xgm4QLVD74qCHLUV45FXQeBBS66LF0vgEAvTCvK2xYf5V7vhKG9R4CdliChat/sPJlmYNywO0wCKMHX/bikSq1YjbaUUjvamNPaqhrjAvrGuzKz9z4+hv/BRMn5efjvktY9yaV5nVYfyMPFeG7bJuzCorMx/IQyyXG++GJx+GhCUz45jiK3Fgb13ch2qE8JM3K1oycGPhy80HCYnHzX6/72hoHCLhimlEePb5HxeGdmSJz4qDIR9y/wa4faUHyybi3hxidpwgvnNTgzEIWPJ1SOF5lUBLJ306wXPTX9DeLwtUKByJ8X9lmv9MmeZ5Eopr0Gne2c19q0dCwJfTRTb71WJEnW/P2+IzZIoAjTiBPAsgpYgomdKtDvKgEUpde40xVSoSz/Av+0/54CPPo9FR3zAR7y4uPogPZ6MLtJi8HV4IORJm4OMsvcuBdnFoOl8mcZvGeNFOWk5ncwx7HGMeRHmjhrMJJfluJz+Gxqxess2qUoPryo8PCjxG/J2LGjCTQKqNZtP4qBk5KiXD6CPMeuvgmi6Ow4MD79O/xfN5uI33ynR+jTGdbi5OL8TOoD/4k4AXP7z6k/j9h1ddPEY9j36Nxr/Exfbrl3/sv/xBdH55d/H+pAfc9Esk3oLMknbF4Q0Qs2ibcg/j/8QonIVZLKs0uKUOJohXPKJwyV4EWO9t1uO9Vf51i0Nmt46PXA3ZrPBo/sQoN4RezOnotJjuBiBXakJOekJXpT6olQy3ulwG9Aezx23r4n4ZqRwv7hydFcZ08MZxmv653ErwmrfOvhpjT3W3L/+1TfeI8lZr8JM0KPjdIzU6LYv9l4MuGz1zn0c5plvg8fgF8mI8u7478CU4w2ecTu8pjTCUxwjHTpUQog1qPHfaNJXlOe9gW27qB/1UUR93Ox9OtDKRyXEjswjjRYC5pS83lN+nIkfuSKQgRMc2LTGIKMsDZlKcRo/Iqet9izm5cLowdjctl5ca+Cbz8fzk+XMhjYUP10/pJERVAs3kUXXR6dPmDEicgOhcJQ/9O01bJlGn2a5DO1flnasg9BXT53PF1vzrHp2JrznjBuupVzCtLKhe1fq81kXZ/FiZHGmfV8cACE8PPhwTRrBY+YZERYeZwSCCZXiPu+1/rimyKAGF5+yX2qsT4pBcYn6VQffsatHhpDGS9RSlSbQj1HLRtefVAukGio4jCXRtBSkjlzjL+4ZcdbmxHS7jKsJKoRNnca1W1cyUkVGaJ7Rr5NJnp8y0XU5IUptZjXQv+ICQF19zWsa6unopV6xnLby1PSToVJYBC3aG+6keSAPoYP9vPqjuHzf/YCWuwOeF4aoyw+1kRr3aLkCamCcUEflYmuSWrLWAgkasvA4U7b1DIajiVN10XNexbn5xVGSYCF6MQbVc0e95QUmypj2XBr5eBy+1BN2viEy68jdthBU1BRRpIzZ9+60xRepZz6lnRAI5AUVrNCAHaNwwrhgP7jAb1naZNbvAB4NInhmF5BmxdTAB2n7KWPAxUBQbeFr1lrMJ6vm7u0vRilSNekVrDeNT2mA9eUrsre1y92iB9rUJyAYwtfKuFaxt/M1I3+q9rttswKE1WeBkRyfAShLRbtXgA0kVfQeZqu7an9z2We5rTwx+k8B7ucE3ObMIxUxIXlorUyP+SYKcBC2Tp7DJfqwkLYMftds4K6+KIzWyTbKWGnMYRe01lULMAm/umeTrGVS8WQ7w0c0odItPdbssndKp7sotUlotR5as5Ug5HhSjHh6dAp8NufJKSTceFX5d2UfzIvdUL127d7pHRH60hJ+e8CwAddvqHlP2aVPG8nch5as676IfIb0qSL2jfnop3qm57vEd5OLFg97jJh0YBCj+vtl9BPnS+LacFJs9MegPunv9ndnjN13xf4VZYrbMsQRUhb98+JhFhqmPwjQCKUIjmDD4s3IdYmoBlejmjz/+sONayEfL8C7J5XUpmgWOW5RxYhTBGAJOXt8UdxH+V+k/tEu5SwcdG3PkctseF0StCeDxTPrBcSjBpBqZnGsHW9rH/zQ4Juq2mvXNFvo5gf/OdgbySLkOmQsJGfJnIsU0ezPJ5jpacxvm4a3Pk0i/HobgoQ6kOwtmZ3LHB6RwqqMvn7ZuHOGrCfawmefFzdaEE1hsi3BM+WFAzg/n+RrLi4Siz21btjBJYBKYA15tzpTFVhr4KH/AJsZ3gIBzQGetzf6bMI9w/NIoJ4v4GDmdnUqDt8MLxz9koQzQM542R/XRmnqy7ekPlSFuoDSUzu5g0CL01RXYVNixDZQ9r3lyVYt1g3lTbAQ+TMof9Ligy430i+7BkWYGa5m04K4VI7lD74VM7qcZbqFnj922yW/SfLjx+UC3QNSIKOizaMWUJ3X/913vddfaWYnxpr7ovOaPm41gwOxWMsWWZCKcUKgnvguza2+wNz6+0GrS6LWdqwjEhTp4j7t45CEUtcCBwoyneAdgKZ0cJHEgG+hZNMNW4LPbgE/8G0f/uXEZ3LqvdcxjxtyuGOyOVv9okSa+AIm6ZWZWbbmSn0v/ZY4SZtM68Z7U94bFwKspqw2A+AH/+2hzAxIYiI5TJoQAAylVkBH+xgwfLCwMnuj6Y+bA5ps62IgScs35BioU76Tpi8h2LhLMaE3SkjRtLSgjPCLNB5WZ81/omhtbskjSO1xEXwCiTPSg0jV4/O/o/vvtce9hApAh3tf5rScmlAd1WdF+f/KHPuynBew8VjrvxD9hnbp1zbjydzLpylWhkvznV1l9ARvnJXxUV3fvOeDyGlzX1QIkaiFW6XrACgPoTZhLf4pNW37jlCp10d/SMUEDBWRLaECB2yhbieFYfGvHLi8nHJRJEc/V7BpWHRvs4GB63E+XRXhH7tQQElDD7KI5OmYegTxDEDD68BgTsPl8HkXLzq7TPa+Tx/mBz7oW89cgUIbkvlGajYKnptUg8MX3Yudxu/5K+egk3FDtSWfiL+lYr89LRyrRplYvLuYRtLgJpeEbcKUGh04WrY6Gr11znsPxqyjYh3Su6BfSBKJNdZYMm0r9N6RIFTouwyxccBjEtyZB2H/Y7G2yHdMeNN4MSolD7Ml4En5W7lDqw6AQ9RL9c0X09u0jBN9WBGz/QU9V2cebW5lfaQ6OxwdzZo+md62Bcqkx/pckTi8EJQ8kklIHrslcs/N7wajA36LfYUkYP/+Xwv2Pp3Ay9VtN5HqCuyW5rSeqOcjfaVnwmOtkij1Oi4j35QWL8R4nsUKJrz9o9nuPyvEiBpGN6aAcBoqzCcmna1gkv0oMUltrT87WKaDtp70aCE4xExxY1nzjVNBBhcWN318jKhkr6ReW1jNIRq/EKWV9BpR2XLTpTK7Yi4cKKI/68nUYhxmqj13b8U1ISDdqtfmtOfdGSHFk0VSqtXPUh+6DKWLwxlXXjLvQCMJKJG5GSjnv3JgyYKJsX4QsEQBgGAjjEGgqKkjyiiUZS8WXIRZWqud/PPZ+FX7pMH4Oeu3uhPJGHXHOTdUUsxoh0ciaEkossuH8PKSSwRDctaXCKm+Z3Ll0DZk3yJW1+oATLZnpZL0YdSQbzkUW3kmsMU8WeoMU6pAfaD0C4YocauhYnhQqvo9P7mOLdJEE9PFVca3WSY/GwFZ1LntzG5iPP8i1MWr1v17Eqhmt+t32d54SawanGntqVcRnc4indL80yf6L8PdgWrL/MsgjvA7sFQCvzYi+dsDozusnBIz+oS1g1I9TxoC01AbadiLT8Vie9ENW2nTajxNYoO2sPa6kowuYWwq6eJ2QDckVwQKrYkh5bytCIqqYRRHO8IDXg93dYy46DxQe1mqQx2dFHMW4nBnBuTu7P8psg+0H68nEV7bljVijc1ojZRcu7bwSTakwtBX337vsKea/f9l7zmfdrp8UX1EfqnAbcms8K6pBm5C1db7W6quhqOQoFaZSmLLCRX8M7P8MWNbsnl1HzOwln0exZs8KpGiwYa84Ijv8PZqUePhUpdzeQkF2HsHWB4rAqfuLLExyun0S750kE/xodCTwPvBKxNb4UC7ly2WUxek0ntSRG3kPj6+i7LBEuXUWJ3F+A930F19ujewF9nFbJ2o4vJPH3xhL9gkiIIXjB13D0QxAuPe9dWytSKuTRbNgMruWZaWQlmHedMxUKm/3AJx9eNQQgeXAun0l/+rGqEqErEppQrCRBJDfyxBPhddjkArpwJ0FCEO+RRxFOc5OzerLvQmbLk7IKfIgsegxeKi7NjcZV+ZrGNZqGiG8ou26ddRHZnT1ZDWFXt1lS65+udm9zst1NrcvzdCzc5Lt9MXPUTG5IUlaJqrfk3YtI2iLxItpFC1lZi0j3eS3mBExjjBYoMNZ9F8OdPFrlpbJNAAWzdkDnRPQVsoyNL1aGHOlD/pgnqfyBDgNi/e6N8UOHWWK8QrLymllhi0jior6XuIalWfPxWVfvKnnMCll9A9GFwdvj0/ftp4lNSbU45Qh1/JOWrkh+UAp4jDuYqXiYQFtMqYRVRusPP7RnjiuIbde3YyxO0gwIyDhpVyLJUVE7w7M7SM/UTBzmFxHHbeWkz4dg2BwzCvHi491XYkcpV3K3ZtU0CtC8Re/1ISPS/GbI3ydXeHPI00TceVFGLXZgG/+3modBffvxU5XfCNeoRbvm9AKa+3uy9lL8XmEjuOKYl6JX8OYzrLjSmmJBSgyjEJC6UuDkaLzgJcFmQMEqbv/uvuIPoxwmePNeZ6j8ZpJGktbh/lbAdWQqJwzlKPwQIYLa2F7NeNacQ7YJ1RqwPWpPg5MZVoLFGSkU6EDEmX3T/CLL8xRthzNuuGDEj5NtNgxcFgF08yXjaGGAhawd4v21YNi66BXA54o26AMKAas6osRTbtOSb3SDOSDEPcTLX1OMfy0rWjrtmaqtc+mTkEE+KLsfz4rmOqlGYvkGlBLX4k8+n3XYpSW2SRyzc0MVuzyPgLtmg4YxehSVXkqcHkJaNLQqGPdXyn6owHfWvxZnzvS0loLmt0r00YD49o1LDFGs6bPymmrUbH1mFBmwWKscW0PBm+LjnbngP/EViMu/3HwupnmYd8UaCzev+mS/xAAzoHzUy3KXuqYFVLbR7koU/WbMp5Pxc8/49VimkY0SRcLgxHlMK8IsRHtKPtKg1D2Xe0jSxlJOTeTVU6ymVMN3gUgFWKNl3p6qNsAxzBxytPtSvyNe3nX3/3hNQD63fDTodFbnAd0Jxtam+Ffjv5UzdIhxAOsob3VDKehvAsOavuuhXMTv2BXrvxetePZj7cBBt3xoa+tyd4tT0e/dY0ueaOkvwX/WL6SLyd/5fJ4+hYg2LXOfjVEWvp7nMfj3wk+3HI20xt2hnAbZfezEFNkan16+nve5LW76P4zJ//6OZPX8Hox5XaN2/nwSmRgH/LXvQGwF0Iehfl0fvC+vhKMzvIjlcRtfPrpaHi4B9OJF8iEolzgZZZVkXFaFPMoAYXGQMUmoMMYgf8WKH4jAG7uyBYi1YFyGrInr6E0XVlZkbUrUykwqMIf9quN3z6AW4a0Vrexf6OZ73gdHd/AVuimtMC3mH4kSKJrSsIfFLlK2P0lCv4WZamvjjqaI9EuXkZ7O065SmuvX18ZZGFaZtItQG74jmZlkZ+w/d1X6Jy3SPLLvjgJywQ0bEmT5TWOUnNFzlqfGFqkSVykWpTtKrbyQ7jDpwYQizT716yJ2aDsbboBO5sP1XRkGATGP9Ap/ua7y0THFDMcoRynKYzrLz+QcwKwgS61RNeI/vH4w7DlskvjSsxxOUPevL9jgJrTZI4DlBua/Dx4rV+wnMg4Cuv9bJl73+fLiA5z46WtdcZUfFzNAcFLV08DI8Ip9HmmPgkOS+3TP/6Lv/BBNZ0LaEleYZz7zZmfV6RJqifKmMzNy3thsdnPO1dqPC2ODmkMpLNPdDmdbNdf46tyNVIDcy8kaGVaYGEsX9tMG0DJk6QGtGny73XHiDZFRMOgfP6qYSYZukXXN4eWVeLhEt0q8YTQTjAYDPD/PoHaqMR7BHhV56ef+oMemnM6A/yjU7W5TS938KWkgV3S1QeSCDa1L6EKPWgwxV9/J2TTaqy5cvh8NYri0xJdqj9ogsGiWwbNopuXd/tNhiXqGZhZsHboErsKNP7nyXalP0+JlnHrroiecSusFQvmViO82auwZmV5wpq9CmdWlpc4s7cSY/B5bMFzUiXU4ayOWjrYLPGkkNdhyXcqRQie32MVuT2xPz7rBVJG0x3XQNcgGVD3eCzbTA3buYuThA6wyOtsu33xcxZFFJkFbTap/fpDnBBwkmXfttNv+KyXgYLavAuBEqpAC38oqv48Jy8CdvMlns9Xjfpp7iDfs9688VnTmdT0tLuS7OcZQMOnPaEEPo1BBvqzIvwN5d91EVyRwj3xoKgHn+7Gw92KPuz1BzM60y3+XYxw86vSRAn8uRaqsZoSKJD4FimBI+fUmPZBO3K0LPVoHHmnmXfbfNVTtN4zTzz61rgL5PJQtMNKQ2TdBQgMSD8aFkqd/feHTXy9PdGBy9Ntib4YCHz4zuBJn7+TLgZa9KDVeI7kodl4Tl+bjedeyKO98dWkCby1mih9CU/U9MRnGH4hsAVYIWu6j1f/vRbshTiEVrLqjASlFeb8WpqttOVIxjJddjy9V9cQ6l29wmsSl/fa1fAyvk0LKdGd7nVdPSG4cwGMGTDSbU4Tbvu0Kgzr2VEnZqSDjKT3mMftas1G8vaUZu0WcsOMTZBDo7k6GlEZzp9qsRCjEM3sdRyPbU5/3eeg/XvHh2Thhnngw8EGB7y9CqZWjz/0CR/18CX0hKYZ5rynuzu1ZXmSxOGXLp4gSTRLDU/MfIxPQ9KeQOobtEbzFLaWe+rZc/EMn7uSp8YpsECgfwq2UXWyGUMLxC79nWPEF37puqHd0zmsYYz5NIDi1u85LpapTpkkblL4tWLCaRlqP/VLjyhR3z5lrV6YVw5F92Q4Q2df/kvMhNGyPmnpOSOGsXkejGgAQQUKipmnJE18xpBAiqGGvtbwcQp/lgO9klFlbrVVSaDq7EA+hyOTGAUuIywRu6vQqQLYf/gs9QNiX9cNugTC5D6Q13TsgyJ538n1RM20AfLqPKrecP82nJdR7rXsSQw222+d6xo5uhq7kWvUZA3TNolXqWvJPqLVHVAgyBr1NJz/HgRn76j1fQFlcI9pr74BIaUpigafNjXGkZOPod09I46mPuFMLlPmLHmT6m3RmJq8Au1quICB+Js8vxpRqG1MMbTpErQa9F7gSHaRPE3k9d9IWWUFxi2bljZbKMh0CRLCDz+68sEqg8aPA3UpOPQvkz6domRAXhBS18yDx3ZIaEsv8F2ykuMj1Y6WVOcp7cDo9tqFEfujTMuL396/IVdi90k9cgj0O+V9PFDR0ciPxcPmv0X5pvRuOw5wPhy+eZqKDnolQUaeY2RHd/Npcz5UyDG5V3N/f/CvweHZ6eHH8/Ph6UXAMY+PEn2BRy/KeREmUVrmIPxKrvykPo9wQ4mT9FqerpS3UuOMh6cHb06GR6LzKcrGaR51NznqkRoEMPBeVJM/Oh7J4u9RyiqAkOL1SvP79aFgoLUu1A36KrNXQ/a2b92cX5zlinLEZulc2Gm6WBBoyClHKdcojZwvvZixi/UdXC1A7d8iicp30zBni1Fhm81r7UihK5P62BJgT6CLeJ+EnF63Lb9P4ab10Sah4lULO96jXWZbPwPdoWY0HarUCocUwltmNWHSUlj7NeWme3g0aRQ4w7OkVAr3PpL3lQo6TwsCZO5JGv3x/OSZAq5VooqCk70G9acGh+fqLp62Lkf1/aw0TW2d1lmTtdbGLtwin2iL55c1JBj0RZbyhvbqG/FDm7ihQciTuVWGVuoChoMA8u6lhgvWOtot2t5w3PWWSj8g/+oncREv9/DGurAs0i2FMPegInzBq6FAqMHz0dAv3S3nQ1lUD1P6uql9wXXdFJR0JMKwFqg3KqKl2G0etiZB2SKkepo0tRfig3OeiC46LcJ7mZaFnD8cESPIhMWH24HtCkmXgRsqFYdps9nH09QGWQZVHyLlBV/01K4hkIaoqxVXDj3Q2/V0ywH+2MuKgvgYOfhkyt4Wd6XubgwLK7GLfbGWWdFwO1Z1tbft1aX7sapIvzkLS2s96Yas6tHvNepV7siqpnzTk/EoK6prSSRrpdlfw3ZmNhAx/1IZPbH+Z7enh+gPiKFJnSTgLKkBpu/uWBu3vk4Tp8wmND0Fv/E5SlCgZckOdsM1yIF4rFCerFCyCCsqQ8pez1waUVUlkCXlApA+gppfDKVGWTCjQsqOTndywlKo2pdtsk5hT2zf+u2Z3L7zRnZg2vK4O2T5ihVJg9wv0f04BYXgOAGyl5VLPQxpLaKM7mMJF6kYxaopdhSXIHGCcHtTFsRJyCBucQPPyXkeb4v9IJ3N0EXmkGdZ03uxYItHy3ZmSVskkLCAoBoE7IwLCAODoHLE2bgJfWw8/n9Ze0N/"
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.
coordinator_url = "" #@param {type:"string"}
drive_folder_name = "" #@param {type:"string"}
#@markdown *Optional*: If Google Drive auto-discovery takes time, paste the Cloudflare or Gradio link from the Main coordinator here (e.g. `https://xyz.trycloudflare.com`), or specify the Google Drive folder name (e.g. `colab drive download`). Leave blank for auto-discovery.
enable_debug_logs = True #@param {type:"boolean"}

import time
import logger
logger.set_debug(enable_debug_logs)
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print(f"   Coordinator URL: {coordinator_url if coordinator_url else 'Auto-Discover (via Google Drive master.json)'}")
if drive_folder_name:
    print(f"   Drive Folder:    {drive_folder_name}")
print(f"   Debug Log Notifications: {'ENABLED (Verbose)' if enable_debug_logs else 'DISABLED (Quiet)'}")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main(coordinator_url=coordinator_url, drive_folder_name=drive_folder_name, enable_debug_logs=enable_debug_logs)
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
